# 🔀 Cross-Model Crosscoder — Gemma-2-2B base vs IT (papergrade)

**What this notebook is.** A reference, paper-grade implementation of an Anthropic-style **cross-model crosscoder**: one shared sparse dictionary trained jointly on residual-stream activations of *two* models on *the same* input tokens. The output is a feature-by-feature **diff** of what model A and model B compute differently.

Companion to `17_train_crosscoder.ipynb` (which is *cross-layer* on a single model). This notebook flips the axis to *cross-model*, follows the 2025–2026 best practice (BatchTopK + science-of-finetuning recipe), and adds a step the literature has not published yet: **causal validation** of shared features.

### Primary sources

1. Lindsey, Templeton, Marcus, Conerly, Batson, Olah (Anthropic). *Sparse Crosscoders for Cross-Layer Features and Model Diffing*, Oct 2024 — <https://transformer-circuits.pub/2024/crosscoders/index.html>
2. Anthropic. *Insights on Crosscoder Model Diffing*, Jan 2025 — <https://transformer-circuits.pub/2025/crosscoder-diffing-update/index.html>
3. Minder, Dumas, Juang, Chughtai, Nanda. *Overcoming Sparsity Artifacts in Crosscoders to Interpret Chat-Tuning*, NeurIPS 2025 — <https://arxiv.org/abs/2504.02922>
4. Bhatt et al. *Cross-Architecture Model Diffing with Crosscoders*, 2026 — <https://arxiv.org/abs/2602.11729>
5. ckkissane/crosscoder-model-diff-replication (open replication) — <https://github.com/ckkissane/crosscoder-model-diff-replication>
6. science-of-finetuning/sparsity-artifacts-crosscoders — <https://github.com/science-of-finetuning/sparsity-artifacts-crosscoders>

### What's different here vs published recipes

- **BatchTopK** (Bussmann 2024) instead of vanilla L1. Minder NeurIPS 2025 §3.3 shows L1 model-diffing crosscoders produce thousands of artifact "chat-only" latents from Complete Shrinkage and Latent Decoupling. We use BatchTopK because its chat-only set is genuinely interpretable.
- **Per-model normalization** (`sqrt(d_model)/mean_norm`) so neither model dominates the MSE term.
- **HF checkpoint resume** every N tokens so a Colab kernel crash costs ≤15 min.
- ⭐ **Causal validation step (§8 below)**. The literature classifies features as *shared* via decoder-norm cosine. Cosine matched ≠ causally equivalent. We ablate each shared feature in both models on the same input, measure the downstream effect, and ask whether the effects correlate. To our knowledge no published crosscoder paper makes this measurement; it is the gap that motivates Paper 1 of the openinterp.org crosscoder program.

### Math (BatchTopK model-diffing crosscoder)

Let $h^A(x), h^B(x) \in \mathbb{R}^{d}$ be the residual-stream activations of models A, B at the same layer on the same input token $x$. Stack them: $\mathbf{h}(x) = [h^A(x); h^B(x)] \in \mathbb{R}^{2 \times d}$.

$$
f_j(x) \;=\; \mathrm{ReLU}\!\left(\sum_{m \in \{A,B\}} (h^m(x) - b^m_{\text{dec}})^\top W^m_{\text{enc},j} + b_{\text{enc},j}\right)
$$

$$
\hat h^A(x) \;=\; \sum_j f_j(x) \cdot d^A_j + b^A_{\text{dec}}, \qquad
\hat h^B(x) \;=\; \sum_j f_j(x) \cdot d^B_j + b^B_{\text{dec}}
$$

**BatchTopK gating**: across a batch of $n$ tokens, retain the top $n \cdot k$ activations of $f_j(x_i) \cdot (\|d^A_j\|_2 + \|d^B_j\|_2)$ globally, zero the rest. At inference, replace by per-feature threshold $\theta_j$ (JumpReLU-style).

**Decoder-norm sparsity loss**:
$$
\mathcal{L} \;=\; \underbrace{\sum_{m \in \{A,B\}} \| h^m - \hat h^m \|_2^2}_{\text{per-model reconstruction}}
\;+\; \lambda \sum_j f_j(x) \big( \|d^A_j\|_2 + \|d^B_j\|_2 \big)
$$

**Δ_norm feature taxonomy** (Lindsey eq. 3, Minder §2.2):
$$
\Delta_{\text{norm}}(j) \;=\; \tfrac{1}{2}\!\left(1 + \frac{\|d^B_j\|_2 - \|d^A_j\|_2}{\max(\|d^A_j\|_2, \|d^B_j\|_2)}\right) \in [0,1]
$$
Thresholds: 0–0.1 → A-only (base); 0.4–0.6 → shared; 0.9–1.0 → B-only (chat).

### Knobs you can change

Default config trains on Gemma-2-2B base vs IT, layer 13, expansion 32 (73,728 latents), 100 M tokens, BatchTopK k=100. Fits an A100 in ~5 h. To scale up: change `CFG["base_model"]`, `CFG["chat_model"]`, `CFG["layer"]`, `CFG["d_model"]`. The notebook is architecture-agnostic — works on any pair of models with matched tokenizer and matched `d_model`.

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors huggingface_hub datasets einops tqdm matplotlib

## 1. Configuration

Single dict so you can scale up by editing one block. Defaults follow Minder Section K (the NeurIPS-2025 paper-grade recipe) for Gemma-2-2B base vs IT.

In [ ]:
import os, math, json, time, hashlib
from pathlib import Path

CFG = {
    # --- model pair (must share tokenizer + d_model) ---
    'base_model':  'google/gemma-2-2b',
    'chat_model':  'google/gemma-2-2b-it',
    'layer':       13,                  # residual-stream layer to hook (Minder default)
    'd_model':     2304,                # Gemma-2-2B hidden size

    # --- crosscoder dictionary ---
    'expansion':   32,                  # n_features = expansion * d_model = 73728
    'k_batchtopk': 100,                 # per-token L0 target after BatchTopK gate
    'k_warmup_init': 1000,              # anneal from 1000 -> k_batchtopk over warmup
    'k_warmup_steps': 5000,             # to prevent dead latents (Minder §K)
    'dec_init_norm': 1.0,               # 1.0 for BatchTopK, 0.05 for L1 (Minder §K — CRITICAL)

    # --- training ---
    'token_budget':  100_000_000,       # 100M tokens (Minder), bump to 400M for Kissane parity
    'seq_len':       512,
    'fwd_batch':     4,                 # LM forward batch (sequences)
    'cc_batch':      4096,              # crosscoder training batch (tokens)
    'lr':            1e-4,              # Minder BatchTopK default
    'lambda_l1':     4.1e-2,            # only used if you switch to L1 variant
    'warmup_steps':  1000,
    'grad_clip':     1.0,
    'lr_decay_frac': 0.20,              # last 20% is linear decay to zero
    'checkpoint_every_tokens': 5_000_000,

    # --- data mix (chat data is essential per Minder; without it chat-only latents underfit) ---
    'data_web_frac':  0.5,
    'data_chat_frac': 0.5,

    # --- publishing ---
    'hf_user':       os.environ.get('HF_USERNAME', 'caiovicentino1'),
    'hf_repo_name':  'gemma2-2b-crosscoder-model-diff-papergrade',
}
CFG['n_features'] = CFG['expansion'] * CFG['d_model']
CFG['hf_repo']    = f"{CFG['hf_user']}/{CFG['hf_repo_name']}"

LOCAL_OUT = Path('/content/crosscoder_modeldiff_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print(f"Pair         : {CFG['base_model']}  vs  {CFG['chat_model']}")
print(f"Layer        : {CFG['layer']}  (d_model = {CFG['d_model']})")
print(f"Dictionary   : {CFG['n_features']:,} latents (expansion {CFG['expansion']})")
print(f"BatchTopK    : k = {CFG['k_batchtopk']}  (anneal {CFG['k_warmup_init']} -> {CFG['k_batchtopk']} over {CFG['k_warmup_steps']:,} steps)")
print(f"Tokens       : {CFG['token_budget']:,}")
print(f"HF repo      : {CFG['hf_repo']}")

## 2. Authenticate & load BOTH models

Both models in **bf16 + SDPA**, frozen. Tokenizer is shared (Gemma family). Memory budget: 2 × 5 GB models + buffer + crosscoder ≈ 12–14 GB at default settings, fits A100-40 with room. On a T4-16 you must drop `cc_batch` to 1024 and `expansion` to 16.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, HfApi, create_repo, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Crosscoder training wants a GPU.'

tok = AutoTokenizer.from_pretrained(CFG['base_model'])
# Sanity: both models must share tokenizer
tok_chat = AutoTokenizer.from_pretrained(CFG['chat_model'])
assert tok.vocab_size == tok_chat.vocab_size, 'Tokenizers diverge — pair only works for shared-vocab models.'

def _load(name):
    print(f'Loading {name} ...')
    m = AutoModelForCausalLM.from_pretrained(
        name,
        dtype=torch.bfloat16,            # transformers 5.x: dtype= (NOT torch_dtype=)
        attn_implementation='sdpa',
        device_map={'': device},
    )
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

model_A = _load(CFG['base_model'])   # base
model_B = _load(CFG['chat_model'])   # chat

def _block_list(m):
    if hasattr(m, 'model') and hasattr(m.model, 'layers'):
        return m.model.layers
    if hasattr(m, 'model') and hasattr(m.model, 'language_model'):
        return m.model.language_model.layers
    raise RuntimeError('Could not locate layer list.')

blocks_A = _block_list(model_A)
blocks_B = _block_list(model_B)
assert len(blocks_A) == len(blocks_B), 'Models must have same depth.'
L = CFG['layer']
assert 0 <= L < len(blocks_A)
print(f'Both models loaded. Hooking layer {L} of {len(blocks_A)}.')

## 3. Paired activation streamer

For every input string we run *both* models and capture the residual at layer L. We mix FineWeb-Edu (web) with UltraChat-200k (chat) at 50/50 — Minder §3 shows that without chat data the chat-specific decoder underfits and you get spurious chat-only artifacts.

We **drop BOS** (Gemma BOS carries pathological norms) and **normalize per-model** by `sqrt(d_model) / mean_activation_norm`. The norm factor is estimated over 100 batches before training and frozen — so neither model dominates the MSE term.

In [ ]:
from datasets import load_dataset
import random

def _web_iter():
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    for row in ds:
        t = row.get('text', '')
        if t and len(t) > 200:
            yield t

def _chat_iter():
    # ultrachat_200k is open; lmsys/lmsys-chat-1m became gated in 2026.
    ds = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft', streaming=True)
    for row in ds:
        msgs = row.get('messages', [])
        if not msgs:
            continue
        text_parts = [m.get('content', '') for m in msgs if m.get('content')]
        t = '\n\n'.join(text_parts)
        if len(t) > 200:
            yield t

def text_stream(web_frac=CFG['data_web_frac']):
    web, chat = _web_iter(), _chat_iter()
    while True:
        if random.random() < web_frac:
            yield next(web)
        else:
            yield next(chat)

class PairedHook:
    """Captures residual-stream output at one layer for one model."""
    def __init__(self, blocks, layer):
        self.buf = None
        self.h = blocks[layer].register_forward_hook(self._hook)
    def _hook(self, _mod, _inp, out):
        h = out[0] if isinstance(out, tuple) else out
        self.buf = h.detach()
    def pop(self):
        b = self.buf; self.buf = None; return b
    def close(self):
        self.h.remove()

@torch.no_grad()
def collect_paired_batch(texts, hookA, hookB):
    enc = tok(texts, return_tensors='pt', max_length=CFG['seq_len'],
              truncation=True, padding='max_length').to(device)
    ids = enc['input_ids']
    mask = enc['attention_mask']
    model_A(ids, attention_mask=mask)
    hA = hookA.pop()                       # (B, T, D) bf16
    model_B(ids, attention_mask=mask)
    hB = hookB.pop()
    # drop BOS (Gemma BOS = position 0)
    hA = hA[:, 1:, :]
    hB = hB[:, 1:, :]
    m = mask[:, 1:].bool()
    # flatten valid tokens
    hA = hA[m].float()                     # (N_valid, D)
    hB = hB[m].float()
    return hA, hB

hookA = PairedHook(blocks_A, L)
hookB = PairedHook(blocks_B, L)

# --- estimate per-model norm scale ---
print('Estimating per-model activation-norm scale (100 batches) ...')
txt = text_stream()
normA_acc, normB_acc, n_acc = 0.0, 0.0, 0
for _ in range(100):
    batch = [next(txt) for _ in range(CFG['fwd_batch'])]
    hA, hB = collect_paired_batch(batch, hookA, hookB)
    normA_acc += hA.norm(dim=-1).mean().item() * hA.shape[0]
    normB_acc += hB.norm(dim=-1).mean().item() * hB.shape[0]
    n_acc += hA.shape[0]
mean_norm_A = normA_acc / n_acc
mean_norm_B = normB_acc / n_acc
norm_scale_A = math.sqrt(CFG['d_model']) / mean_norm_A
norm_scale_B = math.sqrt(CFG['d_model']) / mean_norm_B
CFG['norm_scale_A'] = norm_scale_A
CFG['norm_scale_B'] = norm_scale_B
print(f'Mean ||h_A|| = {mean_norm_A:.3f}  -> scale {norm_scale_A:.4f}')
print(f'Mean ||h_B|| = {mean_norm_B:.3f}  -> scale {norm_scale_B:.4f}')

def paired_stream(buffer_mult=128):
    """Yields (cc_batch, 2, D) float32 minibatches forever, normalized + shuffled."""
    txt = text_stream()
    pending_A, pending_B = [], []
    while True:
        target_rows = CFG['cc_batch'] * buffer_mult
        rows = sum(p.shape[0] for p in pending_A)
        while rows < target_rows:
            batch = [next(txt) for _ in range(CFG['fwd_batch'])]
            hA, hB = collect_paired_batch(batch, hookA, hookB)
            pending_A.append((hA * norm_scale_A).cpu())
            pending_B.append((hB * norm_scale_B).cpu())
            rows += hA.shape[0]
        all_A = torch.cat(pending_A, dim=0)
        all_B = torch.cat(pending_B, dim=0)
        perm  = torch.randperm(all_A.shape[0])
        all_A = all_A[perm]
        all_B = all_B[perm]
        for i in range(0, target_rows - CFG['cc_batch'] + 1, CFG['cc_batch']):
            chunkA = all_A[i:i+CFG['cc_batch']].to(device, non_blocking=True)
            chunkB = all_B[i:i+CFG['cc_batch']].to(device, non_blocking=True)
            yield torch.stack([chunkA, chunkB], dim=1)   # (cc_batch, 2, D)
        leftover = all_A.shape[0] - (target_rows // CFG['cc_batch']) * CFG['cc_batch']
        if leftover > 0:
            pending_A = [all_A[-leftover:]]
            pending_B = [all_B[-leftover:]]
        else:
            pending_A, pending_B = [], []

print('Paired stream ready.')

## 4. BatchTopK Cross-Model Crosscoder

**Architecture**:
- `W_enc: (2, d_model, n_features)` — per-model encoder weights
- `W_dec: (n_features, 2, d_model)` — per-model decoder weights
- `b_enc: (n_features,)` — shared
- `b_dec: (2, d_model)` — per-model

**BatchTopK gate**: across an entire batch of $n$ tokens, retain the top $n \cdot k$ activations of $f_j(x_i) \cdot (\|d^A_j\| + \|d^B_j\|)$. Equivalent to taking top $k$ on average per token, but allows variable per-token sparsity. Bussmann 2024 showed this reduces dead latents and gives cleaner reconstruction-vs-sparsity tradeoffs.

**Init**: encoder = transposed decoder (Minder §K). Decoder norm initialized to `dec_init_norm = 1.0` (critical for BatchTopK convergence).

In [ ]:
from einops import einsum, rearrange

class BatchTopKCrossCoder(nn.Module):
    def __init__(self, d_model: int, n_features: int, k: int, dec_init_norm: float = 1.0):
        super().__init__()
        self.D = d_model
        self.N = n_features
        self.k = k

        # per-model encoder + decoder, shared b_enc, per-model b_dec
        self.W_enc = nn.Parameter(torch.empty(2, d_model, n_features, dtype=torch.float32))
        self.W_dec = nn.Parameter(torch.empty(n_features, 2, d_model, dtype=torch.float32))
        self.b_enc = nn.Parameter(torch.zeros(n_features, dtype=torch.float32))
        self.b_dec = nn.Parameter(torch.zeros(2, d_model, dtype=torch.float32))
        # JumpReLU-style learned threshold (used at inference only, after training)
        self.register_buffer('threshold', torch.zeros(n_features, dtype=torch.float32))
        self._inference_mode = False

        self._init_weights(dec_init_norm)

    def _init_weights(self, dec_init_norm: float):
        # decoder: random direction, fixed L2 norm = dec_init_norm
        with torch.no_grad():
            W = torch.randn_like(self.W_dec)
            W = W / W.norm(dim=-1, keepdim=True).clamp_min(1e-8) * dec_init_norm
            self.W_dec.copy_(W)
            # encoder = transpose of decoder per-model
            #   W_enc[m, d, j]  <-  W_dec[j, m, d]
            self.W_enc.copy_(rearrange(self.W_dec, 'n m d -> m d n'))

    def encode_pre(self, h):
        """h: (B, 2, D) -> pre: (B, N) before activation/gate."""
        h_centered = h - self.b_dec[None]                        # (B, 2, D)
        pre = einsum(h_centered, self.W_enc, 'b m d, m d n -> b n')
        return pre + self.b_enc

    def decoder_norms(self):
        """Returns (n_features, 2): per-feature, per-model decoder L2 norm."""
        return self.W_dec.norm(dim=-1)

    def encode(self, h, k_now=None):
        pre = self.encode_pre(h)                                 # (B, N)
        acts = F.relu(pre)
        if self._inference_mode:
            # JumpReLU at inference: keep f_j if pre_j > threshold_j
            return torch.where(pre > self.threshold[None], acts, torch.zeros_like(acts))
        # BatchTopK at training: rank by acts * (sum of decoder norms)
        dnorm = self.decoder_norms().sum(dim=-1)                 # (N,)
        scaled = acts * dnorm[None]
        k = k_now if k_now is not None else self.k
        topn = k * acts.shape[0]                                 # batch-wide top n*k
        flat = scaled.flatten()
        if topn >= flat.numel():
            return acts
        thr = flat.topk(topn, sorted=False).values.min()
        mask = (scaled >= thr)
        return acts * mask

    def decode(self, z):
        """z: (B, N) -> (B, 2, D)"""
        return einsum(z, self.W_dec, 'b n, n m d -> b m d') + self.b_dec[None]

    def forward(self, h, k_now=None):
        z = self.encode(h, k_now=k_now)
        h_hat = self.decode(z)
        return h_hat, z

    @torch.no_grad()
    def calibrate_threshold(self, stream, n_batches=20):
        """After training, set per-feature threshold so BatchTopK ~= JumpReLU at inference."""
        was_train = self.training
        self.eval()
        self._inference_mode = False
        thrs = []
        for _ in range(n_batches):
            h = next(stream)
            pre = self.encode_pre(h)
            acts = F.relu(pre)
            z = self.encode(h)                                  # BatchTopK
            # for each feature, the smallest pre value that survived BatchTopK
            survived = pre * (z > 0).float()
            survived[survived == 0] = float('inf')
            thrs.append(survived.min(dim=0).values)
        thr = torch.stack(thrs, dim=0).min(dim=0).values
        # cap at 0 so always-firing features don't get a negative threshold
        thr = torch.where(torch.isfinite(thr), thr, torch.zeros_like(thr))
        self.threshold.copy_(thr.clamp_min(0))
        self._inference_mode = True
        if was_train:
            self.train()

n_feat = CFG['n_features']
cc = BatchTopKCrossCoder(
    d_model=CFG['d_model'],
    n_features=n_feat,
    k=CFG['k_batchtopk'],
    dec_init_norm=CFG['dec_init_norm'],
).to(device)
n_params = sum(p.numel() for p in cc.parameters())
print(f'Crosscoder: {n_feat:,} latents · {n_params/1e6:.1f}M params · dec_init_norm = {CFG["dec_init_norm"]}')

## 5. Training loop with HF checkpoint resume

**Schedule**: linear LR warmup over first `warmup_steps`, constant 80%, linear decay-to-zero in last 20%. **k anneal**: BatchTopK k linearly anneals from `k_warmup_init` (1000) → `k_batchtopk` (100) over `k_warmup_steps` to keep latents alive early. **Grad clip**: max-norm 1.0.

**Resume**: every `checkpoint_every_tokens` we push `crosscoder_resume.pt` (state_dict + optim + step + tokens_seen) to HF. On notebook restart, this cell re-downloads and continues. Kernel crash costs ≤ tokens-per-resume.

In [ ]:
from tqdm.auto import tqdm

# --- create HF repo upfront so we can fail-loud here, not at the end ---
api = HfApi()
create_repo(CFG['hf_repo'], exist_ok=True, private=False, token=HF_TOKEN)

opt = torch.optim.Adam(cc.parameters(), lr=CFG['lr'], betas=(0.9, 0.999))

def k_at(step):
    if step >= CFG['k_warmup_steps']:
        return CFG['k_batchtopk']
    frac = step / CFG['k_warmup_steps']
    return int(CFG['k_warmup_init'] - frac * (CFG['k_warmup_init'] - CFG['k_batchtopk']))

def lr_at(step, total_steps):
    if step < CFG['warmup_steps']:
        return CFG['lr'] * (step + 1) / CFG['warmup_steps']
    decay_start = int((1 - CFG['lr_decay_frac']) * total_steps)
    if step < decay_start:
        return CFG['lr']
    prog = (step - decay_start) / max(1, total_steps - decay_start)
    return CFG['lr'] * (1.0 - prog)

total_steps = CFG['token_budget'] // CFG['cc_batch']
log = {'step': [], 'loss': [], 'mse_A': [], 'mse_B': [], 'l1_norm': [], 've_A': [], 've_B': [], 'k': [], 'lr': []}
tokens_seen = 0
step0 = 0

# --- try to resume ---
try:
    ckpt_path = hf_hub_download(repo_id=CFG['hf_repo'], filename='crosscoder_resume.pt', token=HF_TOKEN)
    state = torch.load(ckpt_path, map_location=device)
    cc.load_state_dict(state['cc'])
    opt.load_state_dict(state['opt'])
    step0 = state['step']
    tokens_seen = state['tokens_seen']
    log = state.get('log', log)
    print(f'Resumed at step {step0:,}, tokens_seen {tokens_seen:,}')
except (HfHubHTTPError, FileNotFoundError):
    print('No resume checkpoint — starting fresh.')

stream = paired_stream()
next_ckpt = tokens_seen + CFG['checkpoint_every_tokens']
lambda_l1 = CFG['lambda_l1']

pbar = tqdm(range(step0, total_steps), dynamic_ncols=True, initial=step0, total=total_steps)
for step in pbar:
    h = next(stream)                                              # (B, 2, D), normalized
    k_now = k_at(step)
    lr_now = lr_at(step, total_steps)
    for g in opt.param_groups:
        g['lr'] = lr_now

    h_hat, z = cc(h, k_now=k_now)
    mse_per_model = (h_hat - h).pow(2).sum(dim=-1).mean(dim=0)    # (2,)
    recon = mse_per_model.sum()
    # decoder-norm sparsity term (Anthropic Jan-2025 form)
    dnorm = cc.decoder_norms().sum(dim=-1)                        # (N,)  ||d_A|| + ||d_B||
    l1 = (z.abs() * dnorm[None]).sum(dim=-1).mean()
    loss = recon + lambda_l1 * l1

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cc.parameters(), CFG['grad_clip'])
    opt.step()

    tokens_seen += CFG['cc_batch']

    if step % 50 == 0:
        with torch.no_grad():
            var_per_model = h.var(dim=(0, 2)).clamp_min(1e-8)     # (2,)
            res_per_model = (h_hat - h).var(dim=(0, 2))           # (2,)
            ve = (1 - res_per_model / var_per_model).tolist()
        log['step'].append(step)
        log['loss'].append(float(loss.item()))
        log['mse_A'].append(float(mse_per_model[0].item()))
        log['mse_B'].append(float(mse_per_model[1].item()))
        log['l1_norm'].append(float(l1.item()))
        log['ve_A'].append(ve[0])
        log['ve_B'].append(ve[1])
        log['k'].append(k_now)
        log['lr'].append(lr_now)
        pbar.set_postfix({
            'loss': f'{loss.item():.2f}',
            'VE_A': f'{ve[0]:.3f}', 'VE_B': f'{ve[1]:.3f}',
            'k': k_now, 'tok': f'{tokens_seen/1e6:.1f}M',
        })

    if tokens_seen >= next_ckpt:
        torch.save({
            'cc': cc.state_dict(), 'opt': opt.state_dict(),
            'step': step + 1, 'tokens_seen': tokens_seen, 'log': log, 'cfg': CFG,
        }, str(LOCAL_OUT / 'crosscoder_resume.pt'))
        try:
            api.upload_file(
                path_or_fileobj=str(LOCAL_OUT / 'crosscoder_resume.pt'),
                path_in_repo='crosscoder_resume.pt',
                repo_id=CFG['hf_repo'], repo_type='model', token=HF_TOKEN,
            )
        except Exception as e:
            print(f'Resume upload failed (continuing): {e}')
        next_ckpt += CFG['checkpoint_every_tokens']

# calibrate JumpReLU threshold for inference-time use (§7-§8)
cc.eval()
cc.calibrate_threshold(paired_stream(buffer_mult=8), n_batches=20)
print('Training done. JumpReLU thresholds calibrated.')

## 6. Reconstruction quality + Δ_norm feature taxonomy

Standard checks:
- **Per-model variance explained** on a held-out validation stream. Both should be ≥ 0.85 at our settings.
- **L0** on validation (after JumpReLU, should match `k_batchtopk` ≈ 100).
- **Dead-feature fraction** (latents that never fired in 20 batches).

Then classify every feature by **Δ_norm**:
$$\Delta_{\text{norm}}(j) = \tfrac{1}{2}\!\left(1 + \frac{\|d^B_j\| - \|d^A_j\|}{\max(\|d^A_j\|, \|d^B_j\|)}\right)$$
Thresholds (Minder §2.2): `0–0.1` = **base-only**, `0.4–0.6` = **shared**, `0.9–1.0` = **chat-only**, rest = **unclassified**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

val_stream = paired_stream(buffer_mult=8)
N_VAL_BATCHES = 20

ve_A_acc, ve_B_acc, l0_acc, fired = [], [], [], torch.zeros(n_feat, device=device)
with torch.no_grad():
    for _ in tqdm(range(N_VAL_BATCHES), desc='validation'):
        h = next(val_stream)
        h_hat, z = cc(h)
        var_pm = h.var(dim=(0, 2)).clamp_min(1e-8)
        res_pm = (h_hat - h).var(dim=(0, 2))
        ve_A_acc.append((1 - res_pm[0] / var_pm[0]).item())
        ve_B_acc.append((1 - res_pm[1] / var_pm[1]).item())
        l0_acc.append((z > 0).float().sum(dim=-1).mean().item())
        fired += (z > 0).float().sum(dim=0)

ve_A = float(np.mean(ve_A_acc)); ve_B = float(np.mean(ve_B_acc))
l0   = float(np.mean(l0_acc))
dead_frac = float((fired == 0).float().mean().item())
print(f'VE (base) = {ve_A:.4f}    VE (chat) = {ve_B:.4f}')
print(f'L0         = {l0:.1f}')
print(f'Dead frac  = {dead_frac*100:.2f}%')

# --- Δ_norm taxonomy ---
with torch.no_grad():
    dn = cc.decoder_norms().detach()                               # (N, 2)
norm_A = dn[:, 0].cpu().numpy()
norm_B = dn[:, 1].cpu().numpy()
delta_norm = 0.5 * (1 + (norm_B - norm_A) / np.maximum(norm_A, norm_B).clip(min=1e-8))

label = np.full(n_feat, 'unclassified', dtype=object)
label[delta_norm <= 0.1] = 'base_only'
label[delta_norm >= 0.9] = 'chat_only'
label[(delta_norm >= 0.4) & (delta_norm <= 0.6)] = 'shared'
# also mark dead
fired_np = fired.cpu().numpy()
label[fired_np == 0] = 'dead'

from collections import Counter
ctr = Counter(label.tolist())
print('\nFeature taxonomy:')
for k in ['base_only', 'shared', 'chat_only', 'unclassified', 'dead']:
    pct = 100 * ctr.get(k, 0) / n_feat
    print(f'  {k:14s} {ctr.get(k,0):6d}   ({pct:5.2f}%)')

# --- plot Δ_norm histogram ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(delta_norm[fired_np > 0], bins=80, color='#1f77b4', alpha=0.85)
ax.axvspan(0.0,  0.1, alpha=0.15, color='#ff7f0e', label='base-only')
ax.axvspan(0.4,  0.6, alpha=0.15, color='#2ca02c', label='shared')
ax.axvspan(0.9,  1.0, alpha=0.15, color='#d62728', label='chat-only')
ax.set_xlabel(r'$\Delta_{\mathrm{norm}}$')
ax.set_ylabel('# features')
ax.set_title(f'Crosscoder feature taxonomy — Gemma-2-2B base vs IT, L{CFG["layer"]}')
ax.legend(loc='upper center')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(LOCAL_OUT / 'delta_norm_hist.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Top-activating examples per feature class

For a sample of features per class, find the input tokens with highest activation. Renders chat-only features (what IT learned), shared features (what both compute), and a few base-only features (what was lost to chat-tuning).

Standard sanity check that the dictionary is interpretable. Minder §3 found ~40% of BatchTopK chat-only latents fire on chat-template tokens (`<sot>`, `user`, `model`) — expect to see this.

In [ ]:
TOP_PER_CLASS = 5
K_EXAMPLES = 6
PROBE_BATCHES = 10

# collect a probe set with original token strings + matched activations
probe_tokens, probe_z = [], []
txt = text_stream()
with torch.no_grad():
    for _ in tqdm(range(PROBE_BATCHES), desc='probe collection'):
        batch = [next(txt) for _ in range(CFG['fwd_batch'])]
        enc = tok(batch, return_tensors='pt', max_length=CFG['seq_len'],
                  truncation=True, padding='max_length').to(device)
        ids  = enc['input_ids']
        mask = enc['attention_mask']
        model_A(ids, attention_mask=mask); hA = hookA.pop()[:, 1:, :]
        model_B(ids, attention_mask=mask); hB = hookB.pop()[:, 1:, :]
        m = mask[:, 1:].bool()
        hA = (hA[m].float() * norm_scale_A)
        hB = (hB[m].float() * norm_scale_B)
        h  = torch.stack([hA, hB], dim=1)
        z  = cc.encode(h)                                       # JumpReLU at inference
        # per-token original string
        for bi in range(ids.shape[0]):
            valid = mask[bi, 1:].nonzero(as_tuple=True)[0]
            for pos in valid.tolist():
                probe_tokens.append(tok.decode(ids[bi, pos+1].item()))
        probe_z.append(z.cpu())
probe_z = torch.cat(probe_z, dim=0).numpy()    # (N_probe_tokens, n_feat)
print(f'Probe set: {len(probe_tokens):,} tokens')

def show_top_for_class(cls, n_features=TOP_PER_CLASS, n_examples=K_EXAMPLES):
    feats = np.where(label == cls)[0]
    if len(feats) == 0:
        print(f'  [{cls}] (none)')
        return
    # rank features in this class by sum of |z| over probe set
    by_total = np.argsort(-np.abs(probe_z[:, feats]).sum(axis=0))[:n_features]
    print(f'\n=== {cls.upper()} ({len(feats)} features total) ===')
    for fi in by_total:
        f = feats[fi]
        z_f = probe_z[:, f]
        top = np.argsort(-np.abs(z_f))[:n_examples]
        toks = [(probe_tokens[t][:30].replace('\n', '\\n'), z_f[t]) for t in top]
        rendered = '  '.join(f'{t!r}({z:+.2f})' for t, z in toks)
        print(f'  f{f:6d} | Δ={delta_norm[f]:.2f} | ‖A‖={norm_A[f]:.2f} ‖B‖={norm_B[f]:.2f}')
        print(f'         {rendered}')

show_top_for_class('chat_only')
show_top_for_class('shared')
show_top_for_class('base_only')

## 8. ⭐ Causal validation — the gap in the literature

**The setup.** The Δ_norm taxonomy classifies a feature as *shared* iff its decoder norms in A and B are similar (cosine of decoder rows is high). This is a **representational** equivalence — it says the feature "points the same way" in both models.

**The gap.** Cosine alignment does not imply *causal* equivalence. Two features with high cosine could still have completely different downstream effects: one calibrates style, the other calibrates content. Every published crosscoder paper we surveyed (Lindsey 2024, Anthropic Jan-2025, Minder 2025, Bhatt 2026) measures matched-feature *cosine* / Δ_norm, never matched-feature *causal effect equivalence*.

**The test.** For each shared feature $f_j$:
1. Sample $N$ probe inputs.
2. For each probe input $x$:
   - Compute baseline next-token logits $\ell^A(x), \ell^B(x)$.
   - **Ablate** feature $j$ in model A: subtract $f_j(x) \cdot d^A_j$ from the residual at layer L during the forward pass, get logits $\tilde\ell^A(x)$. Similarly $\tilde\ell^B(x)$.
   - Causal effect in A: $e^A_j(x) = \mathrm{KL}(\mathrm{softmax}(\tilde\ell^A) \,\|\, \mathrm{softmax}(\ell^A))$. Same for B.
3. **Causal equivalence score**: $\mathrm{CE}_j = \mathrm{Pearson}\big(\{e^A_j(x_i)\}, \{e^B_j(x_i)\}\big)$.

If shared features are causally equivalent, CE should concentrate near +1. If cosine ≠ causal, CE distribution will be much flatter than the Δ_norm classification suggests.

**Interpretation:** any feature with high cosine match but `CE < 0.5` is a *cosine-shared but causally-divergent* feature — the literature would call it "universal", we'd say it's an artifact.

In [ ]:
from scipy.stats import pearsonr

# --- pick features to test ---
CAUSAL_PER_CLASS = 80                     # how many features per class to test (cost-vs-coverage knob)
CAUSAL_PROBE_INPUTS = 64                  # token positions per feature

shared_idx = np.where(label == 'shared')[0]
if len(shared_idx) > CAUSAL_PER_CLASS:
    shared_idx = np.random.RandomState(0).choice(shared_idx, CAUSAL_PER_CLASS, replace=False)
# Also test some non-shared features as control — should give CE ≈ 0
control_idx = np.where(label == 'unclassified')[0]
if len(control_idx) > 30:
    control_idx = np.random.RandomState(1).choice(control_idx, 30, replace=False)
test_idx = np.concatenate([shared_idx, control_idx])
print(f'Testing {len(shared_idx)} shared + {len(control_idx)} control features.')

# --- collect probe inputs (token IDs + position to ablate at) ---
probe_ids, probe_pos, probe_baseline_logits_A, probe_baseline_logits_B = [], [], [], []
txt = text_stream()
needed = CAUSAL_PROBE_INPUTS
with torch.no_grad():
    while len(probe_ids) < needed:
        batch = [next(txt) for _ in range(CFG['fwd_batch'])]
        enc = tok(batch, return_tensors='pt', max_length=128, truncation=True, padding='max_length').to(device)
        for bi in range(enc['input_ids'].shape[0]):
            n_valid = enc['attention_mask'][bi].sum().item()
            if n_valid < 32:
                continue
            ids = enc['input_ids'][bi:bi+1, :n_valid]
            pos = n_valid - 1                         # ablate at last token
            outA = model_A(ids).logits[0, pos].float()
            outB = model_B(ids).logits[0, pos].float()
            probe_ids.append(ids.cpu()); probe_pos.append(pos)
            probe_baseline_logits_A.append(outA.cpu())
            probe_baseline_logits_B.append(outB.cpu())
            if len(probe_ids) >= needed:
                break
print(f'Collected {len(probe_ids)} probe inputs.')

# --- ablation hook helper ---
class AblationHook:
    """Subtracts vec from residual at layer L on the last token only."""
    def __init__(self, blocks, layer, pos):
        self.layer = layer; self.pos = pos
        self.vec = None
        self.h = blocks[layer].register_forward_hook(self._hook)
    def set_vec(self, v):
        self.vec = v
    def _hook(self, _mod, _inp, out):
        if self.vec is None:
            return out
        h = out[0] if isinstance(out, tuple) else out
        h = h.clone()
        h[:, self.pos, :] = h[:, self.pos, :] - self.vec.to(h.dtype).to(h.device)
        return (h,) + out[1:] if isinstance(out, tuple) else h
    def close(self):
        self.h.remove()

def kl(p_logits, q_logits):
    """KL(softmax(p) || softmax(q))."""
    p = F.log_softmax(p_logits.float(), dim=-1)
    q = F.log_softmax(q_logits.float(), dim=-1)
    return (p.exp() * (p - q)).sum().item()

# --- causal effect per (feature, probe) ---
ce_records = []
for f_id in tqdm(test_idx, desc='causal validation'):
    eA, eB = [], []
    # decoder vectors for this feature, in unnormalized space
    d_A_unscaled = cc.W_dec[f_id, 0].detach() / norm_scale_A
    d_B_unscaled = cc.W_dec[f_id, 1].detach() / norm_scale_B
    for ids, pos, bl_A, bl_B in zip(probe_ids, probe_pos, probe_baseline_logits_A, probe_baseline_logits_B):
        ids = ids.to(device)
        # 1) get f_j(x) for this input by running encoder on layer-L activations
        with torch.no_grad():
            model_A(ids); hA = hookA.pop()[0, pos].float() * norm_scale_A
            model_B(ids); hB = hookB.pop()[0, pos].float() * norm_scale_B
            h_pair = torch.stack([hA, hB], dim=0).unsqueeze(0)        # (1, 2, D)
            z = cc.encode(h_pair)[0]                                   # (N,)
        f_val = z[f_id].item()
        if f_val <= 0:
            continue   # feature didn't fire; ablation is no-op
        # 2) ablate in model A: subtract f_val * d_A_unscaled at residual layer L
        ablA = AblationHook(blocks_A, L, pos); ablA.set_vec(f_val * d_A_unscaled)
        with torch.no_grad():
            tlA = model_A(ids).logits[0, pos].float().cpu()
        ablA.close()
        ablB = AblationHook(blocks_B, L, pos); ablB.set_vec(f_val * d_B_unscaled)
        with torch.no_grad():
            tlB = model_B(ids).logits[0, pos].float().cpu()
        ablB.close()
        eA.append(kl(tlA, bl_A))
        eB.append(kl(tlB, bl_B))
    if len(eA) >= 8:
        r, _ = pearsonr(eA, eB)
        ce_records.append({
            'feature': int(f_id),
            'class': str(label[f_id]),
            'delta_norm': float(delta_norm[f_id]),
            'cosine': float(F.cosine_similarity(
                cc.W_dec[f_id, 0].detach().unsqueeze(0),
                cc.W_dec[f_id, 1].detach().unsqueeze(0)).item()),
            'mean_KL_A': float(np.mean(eA)),
            'mean_KL_B': float(np.mean(eB)),
            'pearson_CE': float(r),
            'n_probes_fired': int(len(eA)),
        })

import pandas as pd
df = pd.DataFrame(ce_records)
if len(df):
    print('\nCausal-equivalence summary:')
    print(df.groupby('class')['pearson_CE'].describe()[['count', 'mean', '50%', 'min', 'max']])
    df.to_csv(LOCAL_OUT / 'causal_validation.csv', index=False)
    print(f'Saved {LOCAL_OUT / "causal_validation.csv"}')
else:
    print('No causal records — try increasing CAUSAL_PROBE_INPUTS.')

## 9. Cosine vs Causal — the killer figure

Plot the disagreement between the two notions of universality. If decoder cosine equals causal-effect equivalence, points should hug the diagonal. The fraction of "shared by cosine but causally divergent" features (top-left quadrant: cosine high, CE low) is the headline number.

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = {'shared': '#2ca02c', 'unclassified': '#7f7f7f',
            'chat_only': '#d62728', 'base_only': '#ff7f0e'}
    for cls, sub in df.groupby('class'):
        ax.scatter(sub['cosine'], sub['pearson_CE'], s=28, alpha=0.65,
                   color=cmap.get(cls, '#1f77b4'), label=f'{cls} (n={len(sub)})')
    ax.plot([-1, 1], [-1, 1], '--', color='k', alpha=0.4, label='cosine = CE (universality holds)')
    ax.axhline(0.5, ls=':', color='r', alpha=0.5)
    ax.axvline(0.5, ls=':', color='r', alpha=0.5)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel(r'Decoder cosine $\cos(d^A_j, d^B_j)$  (representational match)')
    ax.set_ylabel(r'Pearson CE  (causal effect correlation across probes)')
    ax.set_title('Cosine universality vs causal universality\nGemma-2-2B base vs IT  (the gap nobody published)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(LOCAL_OUT / 'cosine_vs_causal.png', dpi=200, bbox_inches='tight')
    fig.savefig(LOCAL_OUT / 'cosine_vs_causal.pdf', bbox_inches='tight')
    plt.show()

    shared = df[df['class'] == 'shared']
    if len(shared):
        cosine_high  = (shared['cosine'] > 0.7)
        causal_low   = (shared['pearson_CE'] < 0.5)
        gap_frac = float((cosine_high & causal_low).mean())
        print(f'\nAmong shared features, fraction with cosine>0.7 but CE<0.5: {gap_frac:.2%}')
        print(f'Median CE (shared)   : {shared["pearson_CE"].median():.3f}')
        print(f'Median cosine (shared): {shared["cosine"].median():.3f}')
    if 'unclassified' in df['class'].unique():
        ctrl = df[df['class'] == 'unclassified']
        print(f'Median CE (control)  : {ctrl["pearson_CE"].median():.3f}  (should be near 0)')

## 10. Save artifacts + push to HF

Final layout on HF:
- `crosscoder_final.safetensors` — encoder/decoder weights + per-feature JumpReLU thresholds
- `cfg.json` — full config + per-model norm scales (needed at inference)
- `feature_taxonomy.json` — Δ_norm + label per feature
- `causal_validation.csv` — the gap evidence
- `delta_norm_hist.png` — taxonomy histogram
- `cosine_vs_causal.png/.pdf` — the killer figure
- `train_log.json` — training trajectory
- `README.md` — model card

This drop is reproducible from `cfg.json` alone (modulo random seed and dataset shuffling).

In [ ]:
from safetensors.torch import save_file

save_file({
    'W_enc': cc.W_enc.detach().cpu().contiguous(),
    'W_dec': cc.W_dec.detach().cpu().contiguous(),
    'b_enc': cc.b_enc.detach().cpu().contiguous(),
    'b_dec': cc.b_dec.detach().cpu().contiguous(),
    'threshold': cc.threshold.detach().cpu().contiguous(),
}, str(LOCAL_OUT / 'crosscoder_final.safetensors'))

(LOCAL_OUT / 'cfg.json').write_text(json.dumps({
    'architecture':         'cross_model_crosscoder',
    'variant':              'BatchTopK with JumpReLU inference threshold',
    'reference':            'Lindsey 2024 + Anthropic Jan-2025 update + Minder NeurIPS 2025',
    **CFG,
    'val_metrics': {
        've_A': ve_A, 've_B': ve_B, 'L0': l0, 'dead_frac': dead_frac,
    },
    'taxonomy_counts': dict(ctr),
}, indent=2))

(LOCAL_OUT / 'feature_taxonomy.json').write_text(json.dumps({
    'thresholds': {'base_only': [0, 0.1], 'shared': [0.4, 0.6], 'chat_only': [0.9, 1.0]},
    'features': [
        {
            'feature':     int(j),
            'delta_norm':  float(delta_norm[j]),
            'norm_A':      float(norm_A[j]),
            'norm_B':      float(norm_B[j]),
            'fired_count': int(fired_np[j]),
            'label':       str(label[j]),
        } for j in range(n_feat)
    ],
}))

(LOCAL_OUT / 'train_log.json').write_text(json.dumps(log))

readme = f"""---
license: apache-2.0
tags:
  - mechanistic-interpretability
  - sparse-autoencoder
  - crosscoder
  - model-diffing
base_model:
  - {CFG['base_model']}
  - {CFG['chat_model']}
---

# Cross-Model Crosscoder — Gemma-2-2B base vs IT (papergrade)

BatchTopK crosscoder trained on layer {CFG['layer']} residual stream of
`{CFG['base_model']}` and `{CFG['chat_model']}` simultaneously. The dictionary
({CFG['n_features']:,} latents) decomposes both models' activations into shared,
base-specific, and chat-specific features.

## Recipe

- BatchTopK k = {CFG['k_batchtopk']} (annealed from {CFG['k_warmup_init']})
- {CFG['token_budget']/1e6:.0f} M training tokens (FineWeb-Edu + UltraChat-200k, 50/50)
- Per-model normalization, BOS dropped
- Adam lr {CFG['lr']}, decay last 20%, grad clip {CFG['grad_clip']}

## Validation

| | base (A) | chat (B) |
|---|---|---|
| variance explained | {ve_A:.4f} | {ve_B:.4f} |

L0 = {l0:.1f},  dead-feature fraction = {dead_frac*100:.2f}%

## Δ_norm taxonomy

{json.dumps(dict(ctr), indent=2)}

## Causal validation (this artifact's contribution)

Beyond decoder-norm taxonomy, every "shared" feature was tested for causal-effect
equivalence: ablate in both models on matched probe inputs, measure Pearson
correlation of the two KL-shifts. See `causal_validation.csv` and
`cosine_vs_causal.png`. Median causal-equivalence over shared features is in
the figure; this is, to our knowledge, the first time this metric is reported
for a model-diffing crosscoder.

## Citation

- Lindsey et al. 2024 — Sparse Crosscoders for Cross-Layer Features and Model Diffing
- Anthropic Jan 2025 — Insights on Crosscoder Model Diffing
- Minder, Dumas, Juang, Chughtai, Nanda — NeurIPS 2025 (arxiv:2504.02922)
- Bhatt et al. — Cross-Architecture Model Diffing with Crosscoders (arxiv:2602.11729)

## Reproduce

Notebook: `OpenInterpretability/notebooks/17b_crosscoder_model_diff_papergrade.ipynb`
"""
(LOCAL_OUT / 'README.md').write_text(readme)

api.upload_folder(
    folder_path=str(LOCAL_OUT),
    repo_id=CFG['hf_repo'],
    repo_type='model',
    token=HF_TOKEN,
    ignore_patterns=['*.ipynb_checkpoints*', '*resume.pt'],   # don't keep big resume
)
print(f'\n✅ Uploaded to https://huggingface.co/{CFG["hf_repo"]}')

## 11. Where to go from here

1. **Apply this exact notebook to a different model pair.**
   - Cross-stage on Qwen3.5-4B base vs `caiovicentino1/Qwen3.5-4B-mechreward-G3-phaseA-step400` → what features did GRPO add/remove?
   - Cross-arch on Qwen3.6-27B vs Qwen3.6-35B-A3B at matched-depth-fraction → does going dense → MoE preserve features?
   - Cross-precision on Qwen3.6-27B fp16 vs INT4 → what does quantization erase?
   - Cross-mode (advanced): two copies of one reasoning model with the same weights but feed `<think>` rollouts to A, answer rollouts to B → reasoning circuit decomposition.

2. **Strengthen causal validation.** Current test uses zero-ablation, which is OOD. Try [optimal ablation (Li & Janson 2024, arxiv:2409.09951)](https://arxiv.org/abs/2409.09951) and compare the gap. If the cosine-causal gap survives optimal ablation, it's an even cleaner negative-universality result.

3. **Latent Scaling diagnostics** (Minder §3.2). Compute `ν^ε`, `ν^r` per feature to disentangle Complete Shrinkage from Latent Decoupling artifacts. We sidestep this by using BatchTopK; for L1 crosscoders it is mandatory.

4. **Auto-interp.** Run [delphi](https://github.com/EleutherAI/delphi) or Caden Juang's pipeline to label each chat-only / shared / base-only feature with a natural-language description. The Minder paper found 40% of BatchTopK chat-only latents are template-token detectors — replicate that and report the proportion at this layer.

5. **Open question we did not answer:** is `Pearson_CE` a sufficient statistic for causal universality, or do we also need higher-order correlation (e.g., correlation of conditional logit differences token-by-token)? Worth a follow-up notebook.

### Reading queue

- Anthropic Oct 2024 — <https://transformer-circuits.pub/2024/crosscoders/index.html>
- Anthropic Jan 2025 update — <https://transformer-circuits.pub/2025/crosscoder-diffing-update/index.html>
- Minder, Dumas, Juang, Chughtai, Nanda NeurIPS 2025 — <https://arxiv.org/abs/2504.02922>
- Bhatt et al. cross-arch crosscoders — <https://arxiv.org/abs/2602.11729>
- Park et al. non-identifiability of steering vectors — <https://arxiv.org/abs/2602.06801>
- BatchTopK SAEs — <https://arxiv.org/abs/2412.06410>
- ckkissane open replication — <https://github.com/ckkissane/crosscoder-model-diff-replication>
- science-of-finetuning training library — <https://github.com/jkminder/dictionary_learning>